# Extra A · Uma rede neuronal: a ideia do Deep Learning

Na teoria vimos que o **Deep Learning** (aprendizagem profunda) usa **redes neuronais**: camadas de
"neurónios" artificiais, cada um faz uma conta simples e passa o resultado à camada seguinte. Com muitas
camadas e **muitíssimos exemplos**, estas redes reconhecem caras, vozes e texto (é o que está dentro do
ChatGPT, do Google Tradutor ou do desbloqueio facial do telemóvel).

Aqui treinamos uma rede **pequena** (2 camadas de 16 neurónios) com os mesmos pontos da equipa e comparamos
com o Random Forest. Pergunta: **quem precisa de mais exemplos?**


| Sinal | Quer dizer |
| --- | --- |
| ▶️ | Corra a célula: clique nela e prima **Shift + Enter** |
| ✏️ | Edite: troque os `...` pela sua resposta e corra outra vez |
| ✍️ | Responda por escrito, na célula de texto logo abaixo (duplo clique para escrever) |
| ✅ | A célula verifica a sua resposta |

In [ ]:
# Célula 0 · ferramentas da aula: corra sem mudar nada ▶️
import os                                   # os: encontrar ficheiros no computador
import json                                 # json: ler a lista de pontos da equipa
import numpy as np                          # numpy: contas com tabelas de números (uma imagem é uma tabela)
import matplotlib.pyplot as plt             # matplotlib: desenhar mapas e gráficos
from sklearn.ensemble import RandomForestClassifier   # scikit-learn: o modelo Random Forest
from sklearn import metrics                 # scikit-learn: as notas do modelo (acerto, IoU, F1, Kappa, ROC-AUC)

BANDAS = ["azul", "verde", "vermelho", "nir", "swir"]
INDICES = ["ndwi", "mndwi", "ndvi"]
TERRENO = ["hand", "altitude", "declive"]
RAIO = 2                                    # cada ponto marcado vale um quadrado de 5 x 5 píxeis (150 m x 150 m)


def encontrar(nome):
    """Procura um ficheiro de dados na pasta dados/ (ao lado do notebook, ou uma pasta acima)."""
    for pasta in ["dados", os.path.join("..", "dados"), "."]:
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"Não encontrei {nome}. Abra o Jupyter na pasta Workshop_2/2_pratica: "
                            "os dados estão em 2_pratica/dados.")


def carregar(nome):
    """Abre uma imagem (.npz): as 5 bandas em reflectância (0 a 1) e os índices NDWI, MNDWI e NDVI."""
    z = np.load(encontrar(nome))
    img = {banda: z[banda] / 10000 for banda in BANDAS}
    with np.errstate(divide="ignore", invalid="ignore"):
        img["ndwi"] = (img["verde"] - img["nir"]) / (img["verde"] + img["nir"])
        img["mndwi"] = (img["verde"] - img["swir"]) / (img["verde"] + img["swir"])
        img["ndvi"] = (img["nir"] - img["vermelho"]) / (img["nir"] + img["vermelho"])
    img["nuvem"] = z["nuvem"]                   # True onde há nuvem, sombra ou falta de dados
    img["zona"] = z["zona"]                     # True dentro do distrito de Marracuene, fora do mar
    for chave in ["data", "satelite", "epoca"]:
        img[chave] = str(z[chave])
    img["caudal_m3s"] = float(z["caudal_m3s"])
    return img


def juntar_terreno(img, extra):
    """Acrescenta as camadas de terreno (iguais em todas as datas) às características da imagem."""
    for chave in TERRENO:
        img[chave] = extra[chave]
    return img


def cor_natural(img, brilho=0.15):
    """Junta vermelho, verde e azul numa imagem a cores."""
    return np.clip(np.dstack([img["vermelho"], img["verde"], img["azul"]]) / brilho, 0, 1)


def ver_imagem(img, titulo="", agua=(), nao_agua=(), grelha=100):
    """Mostra a imagem a cores com linhas e colunas numeradas, e os pontos marcados (círculo = água, quadrado = não água)."""
    plt.figure(figsize=(9, 12))
    plt.imshow(cor_natural(img))
    for linha, coluna in agua:
        plt.plot(coluna, linha, "o", ms=9, mfc="cyan", mec="black")
    for linha, coluna in nao_agua:
        plt.plot(coluna, linha, "s", ms=9, mfc="orange", mec="black")
    plt.xticks(range(0, img["verde"].shape[1], grelha))
    plt.yticks(range(0, img["verde"].shape[0], grelha))
    plt.grid(alpha=0.5, color="white")
    plt.xlabel("coluna")
    plt.ylabel("linha")
    plt.title(titulo)
    plt.show()


def ver_zona(img, linha, coluna, tamanho=60):
    """Lupa: mostra uma zona pequena à volta de (linha, coluna), com grelha fina, para escolher pontos certos."""
    l0, c0 = max(linha - tamanho, 0), max(coluna - tamanho, 0)
    recorte = cor_natural(img)[l0:linha + tamanho, c0:coluna + tamanho]
    plt.figure(figsize=(7, 7))
    plt.imshow(recorte, extent=(c0 - 0.5, c0 + recorte.shape[1] - 0.5, l0 + recorte.shape[0] - 0.5, l0 - 0.5))
    plt.plot(coluna, linha, "+", ms=20, color="red")
    plt.grid(alpha=0.5, color="white")
    plt.xlabel("coluna")
    plt.ylabel("linha")
    plt.title(f"Lupa à volta da linha {linha}, coluna {coluna}")
    plt.show()


def tabela(img, caracteristicas):
    """Uma linha por píxel e uma coluna por característica: é assim que o modelo lê a imagem."""
    pilha = np.stack([img[c] for c in caracteristicas], axis=-1)
    return np.nan_to_num(pilha.reshape(-1, len(caracteristicas)))


def exemplos(img, agua, nao_agua, caracteristicas, raio=RAIO, grupos=False):
    """Transforma os pontos marcados em exemplos: X (as características) e y (1 = água, 0 = não água).
    raio: cada ponto vale um quadrado de (2 x raio + 1)² píxeis. grupos=True devolve também o número do ponto
    de cada píxel (para a validação cruzada por pontos, no nível avançado)."""
    pilha = np.nan_to_num(np.stack([img[c] for c in caracteristicas], axis=-1))
    X, y, g, nas_nuvens = [], [], [], 0
    for classe, pontos in ((1, agua), (0, nao_agua)):
        for linha, coluna in pontos:
            janela = (slice(max(linha - raio, 0), linha + raio + 1), slice(max(coluna - raio, 0), coluna + raio + 1))
            limpo = ~img["nuvem"][janela]
            if not limpo.any():
                nas_nuvens += 1
                continue
            X.append(pilha[janela][limpo])
            y.append(np.full(limpo.sum(), classe))
            g.append(np.full(limpo.sum(), len(g)))
    if nas_nuvens:
        print(f"Atenção: {nas_nuvens} ponto(s) caíram em nuvens ou fora da imagem e foram ignorados.")
    if grupos:
        return np.concatenate(X), np.concatenate(y), np.concatenate(g)
    return np.concatenate(X), np.concatenate(y)


def prever(modelo, img, caracteristicas):
    """A probabilidade de ser água (0 a 1), para cada píxel da imagem."""
    probabilidade = modelo.predict_proba(tabela(img, caracteristicas))[:, 1]
    return probabilidade.reshape(img["verde"].shape)


def notas(verdade, previsto, probabilidade=None):
    """As notas do modelo comparado com a verdade (1 = água, 0 = não água)."""
    r = {"acerto": metrics.accuracy_score(verdade, previsto),
         "IoU": metrics.jaccard_score(verdade, previsto),
         "F1": metrics.f1_score(verdade, previsto),
         "Kappa": metrics.cohen_kappa_score(verdade, previsto)}
    if probabilidade is not None:
        r["ROC-AUC"] = metrics.roc_auc_score(verdade, probabilidade)
    return r


def mostrar_notas(nome, r):
    print(f"{nome:28s}", "  ".join(f"{k} {v:.3f}" for k, v in r.items()))


def contar_erros(verdade, previsto):
    """A matriz de confusão em palavras: o que acertou e o que errou."""
    agua_certa = int(((verdade == 1) & (previsto == 1)).sum())
    terra_certa = int(((verdade == 0) & (previsto == 0)).sum())
    falso_alarme = int(((verdade == 0) & (previsto == 1)).sum())
    agua_perdida = int(((verdade == 1) & (previsto == 0)).sum())
    print(f"Água que o modelo encontrou (certo):      {agua_certa:8d} píxeis")
    print(f"Terra que o modelo disse terra (certo):   {terra_certa:8d} píxeis")
    print(f"Falsos alarmes (terra que disse água):    {falso_alarme:8d} píxeis")
    print(f"Água perdida (água que disse terra):      {agua_perdida:8d} píxeis")
    return agua_certa, terra_certa, falso_alarme, agua_perdida


print("Ferramentas prontas.")

In [ ]:
from sklearn.neural_network import MLPClassifier      # uma rede neuronal (perceptrão multicamada)
from sklearn.pipeline import make_pipeline            # encadear passos
from sklearn.preprocessing import StandardScaler      # pôr todas as características na mesma escala (as redes precisam)

cheia = carregar("incomati_2026_cheia.npz")
extra = dict(np.load(encontrar("camadas_extra.npz")))
equipa = json.load(open(encontrar("exemplos_da_equipa.json")))
CARACTERISTICAS = BANDAS + INDICES
X, y = exemplos(cheia, equipa["2026"]["agua"], equipa["2026"]["nao_agua"], CARACTERISTICAS)
teste = (extra["ems_2026"] >= 0) & ~cheia["nuvem"]
verdade = extra["ems_2026"][teste]
print("Exemplos de 2026:", len(y), "píxeis")

## Passo 1 · Treinar os dois modelos só com 2026 ▶️

In [ ]:
def treinar_os_dois(X, y):
    rede = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=800, random_state=0))
    floresta = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
    return {"Rede neuronal": rede.fit(X, y), "Random Forest": floresta.fit(X, y)}


def exame(modelos):
    for nome, m in modelos.items():
        p = m.predict_proba(tabela(cheia, CARACTERISTICAS))[:, 1].reshape(cheia["verde"].shape)
        linha = f"{nome:14s} IoU no exame EMS 2026: {notas(verdade, (p[teste] > 0.5).astype(int))['IoU']:.3f}"
        for ano in [2021, 2024]:
            img = carregar(f"incomati_{ano}_chuvosa.npz")
            pa = m.predict_proba(tabela(img, CARACTERISTICAS))[:, 1].reshape(img["verde"].shape)
            linha += f" | {ano}: encontra {100 * (pa[extra['verif_agua'] & ~img['nuvem']] > 0.5).mean():3.0f}% do rio"
        print(linha)


exame(treinar_os_dois(X, y))

✍️ **Pergunta:** em 2026 as duas empatam. E em 2024 (água muito escura)? Qual generaliza melhor com poucos exemplos?

*Resposta:*

## Passo 2 · Dar exemplos de todos os anos ▶️

Juntamos os 5 + 5 pontos da equipa de cada um dos anos 2014, 2021, 2022, 2024 e 2025.

In [ ]:
Xs, ys = [X], [y]
for ano in [2014, 2021, 2022, 2024, 2025]:
    img = carregar(f"incomati_{ano}_chuvosa.npz")
    xa, ya = exemplos(img, equipa[str(ano)]["agua"], equipa[str(ano)]["nao_agua"], CARACTERISTICAS)
    Xs.append(xa)
    ys.append(ya)
X_todos, y_todos = np.concatenate(Xs), np.concatenate(ys)
print("Exemplos de todos os anos:", len(y_todos), "píxeis")
exame(treinar_os_dois(X_todos, y_todos))

## O que aprendemos

- Com **poucos exemplos e pouca variedade**, a rede neuronal aprende uma regra frágil e falha nos casos que nunca viu.
- Com **exemplos variados**, a rede e o Random Forest ficam parecidos, e **os dois melhoram** no exame de 2026.
- É por isso que o Deep Learning a sério usa **milhares ou milhões** de exemplos marcados por pessoas. Por exemplo,
  o mapa mundial de água e uso do solo **Dynamic World** (Google), que o projecto ETE usa, é uma rede neuronal treinada
  com dezenas de milhares de imagens marcadas à mão.
- Para poucos exemplos, o **Random Forest** é muitas vezes a melhor escolha: rápido, robusto e fácil de explicar.